In [36]:
from dataclasses import dataclass
 
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
from models.vit_gpt2_model import ViTGPT2FromScratch

In [3]:
m = ViTGPT2FromScratch()
px = torch.randn(2, 3, 224, 224)
ids = torch.randint(0, 50257, (2, 12))
out = m(px, ids, labels=ids.clone())
print("logits", tuple(out["logits"].shape), "loss", float(out["loss"]))
gen = m.generate(px, bos_id=50256, eos_id=50256, max_len=10, temperature=0.0)
print("generated", tuple(gen.shape))
print("params(M)", sum(p.numel() for p in m.parameters()) / 1e6)

/tmp/ipykernel_34501/3106291990.py:5: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:852.)
  print("logits", tuple(out["logits"].shape), "loss", float(out["loss"]))


logits (2, 12, 50257) loss 10.796475410461426
generated (2, 11)
params(M) 239.19744


In [4]:
from models.vit_gpt2_model import build_vit_gpt2_for_finetune, build_vit_gpt2_pretrained, caption

In [5]:
model_pretrained, img_processor_pretrained, tokenizer_pretrained = build_vit_gpt2_pretrained(device=device)

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from pathlib import Path
from PIL import Image

paths = sorted(Path('data/samples').glob("*.jpg"))
imgs = [Image.open(p).convert("RGB") for p in paths]

In [9]:
caption_pretrained = caption(model=model_pretrained, image_processor=img_processor_pretrained, tokenizer=tokenizer_pretrained, images=imgs, device=device)

In [13]:
import matplotlib.pyplot as plt
import textwrap

def compare_captioned(images, captions, cols=3, scale=4):
    n = len(images)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(scale * cols, scale * rows), squeeze=False)
    axes = axes.ravel()
    for ax, img, cap in zip(axes, images, captions):
        ax.imshow(img)
        ax.set_title(textwrap.fill(cap, 30), fontsize=10)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("big_grid.png", dpi=100, bbox_inches="tight")
    plt.close()

___

# BLIP Transfer Learning

In [1]:
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import textwrap

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

device

'cuda'

In [6]:
from models.blip_based_model import BLIPCaptioner

In [9]:
blip = BLIPCaptioner(device=device, dtype=dtype)

blip.model.eval()

print("params(M)", sum(p.numel() for p in blip.model.parameters()) / 1e6)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

params(M) 223.971644


In [10]:
paths = sorted(Path("data/samples").glob("*.jpg"))
imgs = [Image.open(p).convert("RGB") for p in paths]

len(imgs), paths[:3]

(28,
 [PosixPath('data/samples/a_man_and_two_women_taking_picture.jpg'),
  PosixPath('data/samples/a_toilet.jpg'),
  PosixPath('data/samples/baseball_player_on_the_field.jpg')])

In [11]:
caption_blip = blip.caption(
    imgs,
    max_new_tokens=30,
    num_beams=3,
)

caption_blip

In file included from /usr/include/python3.14/pyconfig.h:6,
                 from /usr/include/python3.14/Python.h:14,
                 from /tmp/tmpgnbvnsjb/cuda_utils.c:9:
/usr/include/python3.14/pyconfig-64.h:2007:9: warning: ‘_POSIX_C_SOURCE’ redefined
 2007 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/sepehr/Templates/python/image-captioning-model/.venv/lib/python3.14/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /tmp/tmpgnbvnsjb/cuda_utils.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~


['three people posing for a picture',
 'a white toilet bowl',
 'a baseball player running to first base',
 'a basketball ball sitting on the ground with palm trees in the background',
 'a small bird perched on a branch of a tree',
 'a blue sports car parked in front of a building',
 'a boy on a skateboard',
 'a plate of food with a hamburger and fries',
 'a cat laying on top of a purple blanket',
 'the ceiling in the vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican vatican',
 'a red race car parked in front of a building',
 'two zebras standing next to each other zebra',
 'a refrigerator in a kitchen',
 'a cup of coffee and a book on a bed',
 'a man sitting on the side of a road next to a bike',
 'a man in a red uniform riding a brown horse',
 'a man sitting on a bench in a field',
 'a skateboarder doing a trick on a rail',
 'a stuffed animal sitting on the back of a boat',
 'the ruins of the ancient city of

In [ ]:
compare_captioned(images=imgs, captions=caption_blip)